# Lab A — CPU vs GPU Inference Baseline / CPU 与 GPU 推理性能基线
## Slide 33 · 同一个模型，两种 AMD 设备

**What will you learn? / 本节学什么？**
- Latency (P50/P95)、Throughput、Warm-up 的准确含义 What latency P50/P95, throughput and warm-up really mean
- 同一份官方 YOLO26l 在 AMD CPU 与 W7900 GPU 上的真实差异 The measured difference of the same official YOLO26l on the AMD CPU vs the W7900

**One key idea / 一个核心原理**
> 深度学习推理 = 海量矩阵乘法。GPU 的数千个并行核心天然适合这类工作负载，但“快多少”必须实测。Inference = huge matrix math. GPUs are built for it, but *how much faster* must be measured, not assumed.

In [1]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

repo: /workspace/yolo26-openvino-rocm-day2-labs


## 1. Key metrics / 关键指标
- **Latency 延迟**: 处理 1 张图的耗时（P50 = 中位数，P95 = 95 分位，反映最差情况）
- **Throughput 吞吐**: 每秒处理多少张图（img/s）
- **Warm-up 预热**: 前几次运行会初始化内核/缓存，不计入统计
- 我们测量的是**端到端**时间：预处理 + 模型计算 + 后处理

Latency = time for one image (P50 median, P95 tail). Throughput = images/s. Warm-up runs initialize kernels and are excluded. We measure **end-to-end** time: preprocess + model + postprocess.

In [2]:
import torch
from course.model import load_official_yolo26l
from course.benchmark import load_bench_images, pin_torch_threads, cgroup_cpu_quota

# Fair CPU config: PyTorch is fastest at 8 threads under this container's
# 16-core CPU quota (measured: 8->190ms, 64->516ms per image!)
print("torch CPU threads:", pin_torch_threads(8),
      "| cgroup CPU quota:", cgroup_cpu_quota(), "cores")
model = load_official_yolo26l()
images = load_bench_images()
IMG = images[:20]          # fixed image pool for this lab

torch CPU threads: 8 | cgroup CPU quota: 16 cores


## 2. Run the baseline: AMD CPU / 基线：AMD CPU
对同一张图片反复推理，取统计值。We repeatedly predict the same images and collect per-iteration latency samples.

In [3]:
WARMUP, ITERS = 3, 10          # quick mode; run_all --profile full uses 10/50

from course.benchmark import bench, save_run, RunMeta

def predict_cpu(path):
    model.predict(source=str(path), imgsz=640, device="cpu",
                  verbose=False)

cpu_ms = bench(lambda: predict_cpu(IMG[0]), warmup=WARMUP, iters=ITERS, sync=False)
cpu_sum = save_run(RunMeta("labA_cpu_vs_gpu", "pytorch_cpu", "cpu",
                           "pytorch-rocm-cpu", warmup=WARMUP, iters=ITERS,
                           notes="notebook quick run"), cpu_ms)
print(f"CPU P50 {cpu_sum['stats']['p50_ms']:.1f} ms · "
      f"P95 {cpu_sum['stats']['p95_ms']:.1f} ms")

[saved] results/raw/labA_cpu_vs_gpu/labA_cpu_vs_gpu_pytorch_cpu_20261108_112741.csv  (10 samples)
CPU P50 186.3 ms · P95 191.5 ms


## 3. Change one thing: same model, same image → W7900 GPU / 只改一件事：换到 W7900
唯一变化是 `device="cuda"`（ROCm GPU）。GPU 计算是异步的，计时前后必须 `torch.cuda.synchronize()`——这正是共享工具 `bench()` 做的事。
The only change: `device="cuda"`. GPU calls are asynchronous, so the shared `bench()` helper synchronizes before/after each timed call.

In [4]:
gpu_ms = bench(lambda: model.predict(source=str(IMG[0]), imgsz=640,
                                device="cuda", verbose=False),
               warmup=WARMUP, iters=ITERS, sync=True)
gpu_sum = save_run(RunMeta("labA_cpu_vs_gpu", "pytorch_rocm_gpu", "cuda(gfx1100)",
                           "pytorch-rocm", warmup=WARMUP, iters=ITERS,
                           notes="notebook quick run"), gpu_ms)
print(f"GPU P50 {gpu_sum['stats']['p50_ms']:.1f} ms · "
      f"P95 {gpu_sum['stats']['p95_ms']:.1f} ms")

[saved] results/raw/labA_cpu_vs_gpu/labA_cpu_vs_gpu_pytorch_rocm_gpu_20261108_112744.csv  (10 samples)
GPU P50 12.9 ms · P95 13.7 ms


## 4. Compare the results / 对比真实结果

In [5]:
import pandas as pd

rows = []
for name, s in [("AMD CPU", cpu_sum), ("W7900 GPU", gpu_sum)]:
    st = s["stats"]
    rows.append({"device": name, "P50 (ms)": round(st["p50_ms"], 1),
                 "P95 (ms)": round(st["p95_ms"], 1),
                 "throughput (img/s)": round(1000 / st["p50_ms"], 1)})
df = pd.DataFrame(rows)
ratio = cpu_sum["stats"]["p50_ms"] / gpu_sum["stats"]["p50_ms"]
df["speedup vs CPU"] = ["1.00x", f"{ratio:.2f}x"]
display(df)

,device,P50 (ms),P95 (ms),throughput (img/s),speedup vs CPU
0,AMD CPU,186.3,191.5,5.4,1.00x
1,W7900 GPU,12.9,13.7,77.6,14.46x


In [6]:
from course.charts import chart_cpu_vs_gpu
chart_cpu_vs_gpu(cpu_sum, gpu_sum)

[chart] /workspace/yolo26-openvino-rocm-day2-labs/assets/slides/01_cpu_vs_gpu.png (+svg)


## 5. Why? / 为什么会这样？
YOLO26l 每张图要执行数千万次乘加。W7900 拥有 6144 个流处理器和 48GB 高带宽显存，可以同时执行成千上万路运算；CPU 虽然单核更强，但并行宽度小得多。
注意：我们比较的是**设备整体方案**（CPU 平台 vs GPU 平台），而不是纯粹软件优化——那是 Lab B 的主题。

YOLO26l performs tens of millions of MACs per image. The W7900 has 6144 stream processors and 48 GB of high-bandwidth memory; the CPU has far fewer but smarter cores. Note we compare **platforms**, not pure software runtimes — that is Lab B.

如果差距没有你想象的极端，想想：预处理/后处理仍在 CPU 上，小 batch 下 GPU 可能“吃不饱”。If the gap is smaller than expected, remember pre/post-processing still runs on the CPU and batch=1 may under-utilize the GPU.

## 5b. Mini-lab: Amdahl's Law, compute it yourself / 迷你实验：Amdahl 定律，自己算

**先别运行代码，把答案写在纸上**：一次完整请求中，CPU 前后处理固定占 20 ms，GPU 模型计算占 10 ms，总耗时 30 ms。
如果 GPU 计算快了 2 倍（10 ms → 5 ms），总耗时是多少？为什么不是 15 ms？
**Write your answer BEFORE running**: a request spends a fixed 20 ms in CPU pre/post-processing and 10 ms in GPU compute (30 ms total). If GPU compute gets 2x faster (10 → 5 ms), what is the new total — and why is it NOT 15 ms?


In [7]:
# Amdahl mini-lab — run AFTER writing your guess / 写完猜测后再运行
STUDENT_AMDAHL_GUESS_MS = None      # <-- 你的答案: 15? 20? 25? / your guess

cpu_fixed_ms = 20.0                 # CPU 前后处理(固定) / fixed CPU pre+post
gpu_ms_before = 10.0                # GPU 模型计算 / GPU model compute
total_before = cpu_fixed_ms + gpu_ms_before
gpu_ms_after = gpu_ms_before / 2    # GPU 加速 2 倍 / GPU 2x faster
total_after = cpu_fixed_ms + gpu_ms_after

print(f"before: {cpu_fixed_ms:.0f} + {gpu_ms_before:.0f} = {total_before:.0f} ms")
print(f"after : {cpu_fixed_ms:.0f} + {gpu_ms_after:.0f} = {total_after:.0f} ms")
print(f"your guess: {STUDENT_AMDAHL_GUESS_MS} ms")
print(f"GPU part is 2x faster, but the SYSTEM only went "
      f"{total_before:.0f}->{total_after:.0f} ms "
      f"({total_before/total_after:.2f}x) — the fixed CPU part caps the gain.")
print(f"Amdahl limit: even an INFINITELY fast GPU cannot beat "
      f"{cpu_fixed_ms:.0f} ms (max {1/(cpu_fixed_ms/total_before):.2f}x).")


before: 20 + 10 = 30 ms
after : 20 + 5 = 25 ms
your guess: None ms
GPU part is 2x faster, but the SYSTEM only went 30->25 ms (1.20x) — the fixed CPU part caps the gain.
Amdahl limit: even an INFINITELY fast GPU cannot beat 20 ms (max 1.50x).


In [8]:
# Where is the fixed overhead in OUR real measurements? / 我们的实测里固定开销在哪？
# Compare end-to-end GPU latency (above) with MODEL-ONLY forward time:
# 对比 GPU 端到端延迟（上面测的）与"纯模型前向"时间：
from course.pipeline import Yolo26lStages
from course.benchmark import pct
stages = Yolo26lStages(imgsz=640)
im = stages.preprocess(stages.load(IMG[0]))
fwd_ms = [ (stages.infer(im), torch.cuda.synchronize())[1] for _ in range(1) ]
import time
fwd = []
for _ in range(5):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    stages.infer(im); torch.cuda.synchronize()
    fwd.append((time.perf_counter() - t0) * 1000)
print(f"GPU end-to-end P50 (measured above): {gpu_sum['stats']['p50_ms']:.1f} ms")
print(f"GPU model-only forward P50         : {pct(fwd, 50):.1f} ms")
print(f"fixed non-model overhead           : "
      f"{gpu_sum['stats']['p50_ms'] - pct(fwd, 50):.1f} ms "
      "(CPU pre/post + data movement / CPU前后处理+数据搬运)")


GPU end-to-end P50 (measured above): 12.9 ms
GPU model-only forward P50         : 9.5 ms
fixed non-model overhead           : 3.4 ms (CPU pre/post + data movement / CPU前后处理+数据搬运)


结论 / Conclusion: GPU 端到端延迟中有一部分**与模型计算无关**（预处理、后处理、数据搬运）。
这与 Slide 30 直接衔接：优化哪一段，收益上限就由哪一段决定——这就是 Amdahl 定律的实测版本。
Part of the end-to-end GPU latency is **not model compute** (pre/post-processing, data movement).
This connects directly to Slide 30: the part you do NOT optimize sets the ceiling of your speedup — Amdahl's law, measured.


## 6. Try it yourself / 学生练习
先写下你的预测，再把 `STUDENT_DEVICE` 改成 `"cpu"` 或 `"cuda"`、`STUDENT_ITERS` 改成 20，重新运行下面格子，观察 P50/P95 的变化。
Predict first, then set `STUDENT_DEVICE` and `STUDENT_ITERS` and re-run.

In [9]:
STUDENT_DEVICE = "cuda"        # <-- change me: "cpu" or "cuda"
STUDENT_ITERS  = 10           # <-- change me
my_prediction  = "P50 will be ___ ms"   # <-- write your prediction

ms = bench(lambda: model.predict(source=str(IMG[1]), imgsz=640,
                     device=STUDENT_DEVICE, verbose=False),
           warmup=2, iters=STUDENT_ITERS, sync=(STUDENT_DEVICE == "cuda"))
from course.benchmark import stats_of
print("prediction :", my_prediction)
print("measured   :", {k: round(v, 1) for k, v in stats_of(ms).items()})

prediction : P50 will be ___ ms
measured   : {'n': 10, 'mean_ms': 11.7, 'p50_ms': 11.7, 'p95_ms': 11.7, 'min_ms': 11.6, 'max_ms': 11.7}


## Common mistakes / 常见错误
- 把 P50 当成唯一指标：P95 反映长尾，线上服务两个都要看。*P50 alone is not enough — P95 shows the tail.*
- 忘记 warm-up：首次推理包含内核编译/缓存，会严重高估延迟。*Skip warm-up and first-iteration JIT costs pollute results.*
- GPU 计时不同步：ROCm 异步执行，计时不加 `torch.cuda.synchronize()` 会得到假的小数字。*Timing GPU work without synchronization gives fake-small numbers.*


**Takeaway / 一句话总结**
> 硬件架构决定计算方式：同一份 YOLO26l，在 GPU 与 CPU 上的执行速度可以相差一个数量级——但具体多少，必须实测。
> Architecture decides how math is executed: the same YOLO26l can run an > order of magnitude differently across devices — measure it.